<a href="https://colab.research.google.com/github/SuchandanG/Optimized-Secure-Admissible-Partitioning-of-Affine-Manifolds-for-General-Access-Structures/blob/main/Four_Party_Optimized_SAP_verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Partitiong verification on 87 Access Stucture

In [4]:
from itertools import combinations
import pandas as pd


# =========================================================
# Local intersection
# =========================================================

def local_intersection(subfamily):

    return set.intersection(*map(set, subfamily))


# =========================================================
# Quotient residuals
# =========================================================

def quotient_residuals(subfamily, S):

    return [set(A) - S for A in subfamily]


# =========================================================
# N-SC condition
# =========================================================

def satisfies_kSC(residuals):

    k = len(residuals)

    for i in range(k):

        ok = True

        for j in range(k):

            if i == j:

                continue

            if residuals[i].intersection(residuals[j]):

                ok = False

                break

        if ok:

            return True

    return False


# =========================================================
# XOR-induced support over F2
# =========================================================

def xor_support(subfamily):

    count = {}

    for A in subfamily:

        for x in A:

            count[x] = count.get(x, 0) + 1

    B = set()

    for x in count:

        if count[x] % 2 == 1:

            B.add(x)

    return frozenset(B)



# =========================================================
# PARITY CLOSURE OF A BLOCK
# =========================================================

def closure_sets(block):

    derived = set()

    n = len(block)

    for k in range(3, n + 1, 2):

        for sub in combinations(block, k):

            X = xor_support(sub)

            if X not in block:
                derived.add(X)

    return derived


# =========================================================
# MONOTONE QUALIFICATION CHECK
# =========================================================

def monotone_qualified(B, Gamma_0):

    for A in Gamma_0:

        if set(A).issubset(set(B)):

            return True

    return False


# =========================================================
# PROPERTY A CHECK
# =========================================================

def admissible_family(family, Gamma_0):

    n = len(family)

    for k in range(3, n + 1, 2):

        for sub in combinations(family, k):

            S = local_intersection(sub)

            residuals = quotient_residuals(sub, S)

            # -------------------------------------------------
            # Property A failure
            # -------------------------------------------------

            if not satisfies_kSC(residuals):

                B = xor_support(sub)

                # -------------------------------------------------
                # Property B admissibility
                # -------------------------------------------------

                if not monotone_qualified(B, Gamma_0):

                    return False

    return True


# =========================================================
# CHECK WHETHER B IS GENERATED
# =========================================================

def generated_by_family(B, family):

    n = len(family)

    for k in range(3, n + 1, 2):

        for sub in combinations(family, k):

            X = xor_support(sub)

            if X == B:

                return True

    return False


# =========================================================
# MINIMAL AFFINE GENERATING BLOCK
# =========================================================

def reduce_block(block):

    generators = list(block)

    derived = set()

    changed = True

    while changed:

        changed = False

        for B in generators.copy():

            others = [
                A for A in generators
                if A != B
            ]

            if generated_by_family(B, others):

                generators.remove(B)

                derived.add(B)

                changed = True

                break

    return generators, derived


# =========================================================
# SUPPORT OF BLOCK
# =========================================================

def block_support(block):

    S = set()

    for A in block:

        S |= set(A)

    return S


# =========================================================
# Encoding COST
# =========================================================

def EC_cost(partition):

    total = 0

    for block in partition:

        total += len(block_support(block))

    return total


# =========================================================
# FORMAT BLOCK
# =========================================================

def format_block(block):

    formatted = []

    for A in sorted(block, key=lambda x: sorted(list(x))):

        formatted.append(
            "{" + ",".join(map(str, sorted(A))) + "}"
        )

    return formatted


# =========================================================
# GENERATE ALL SET PARTITIONS
# =========================================================

def all_partitions(collection):

    if len(collection) == 1:

        yield [collection]

        return

    first = collection[0]

    for smaller in all_partitions(collection[1:]):

        # insert into existing block

        for n, subset in enumerate(smaller):

            yield (
                smaller[:n]
                + [[first] + subset]
                + smaller[n + 1:]
            )

        # create singleton block

        yield [[first]] + smaller


# =========================================================
# CHECK GLOBAL SAP PARTITION
# =========================================================

def admissible_partition(partition, Gamma_0):

    reduced_partition = []
    derived_per_block = []

    for block in partition:

        if not admissible_family(block, Gamma_0):
            return False, [], set()


        reduced, _ = reduce_block(block)

        derived = closure_sets(reduced)

        reduced_partition.append(reduced)
        derived_per_block.append(derived)

# --------------------------------------------
# NEW CHECK
# Derived sets cannot appear in another block
# --------------------------------------------

    for i in range(len(reduced_partition)):

        for j in range(len(reduced_partition)):

            if i == j:
                continue

            for D in derived_per_block[i]:

                if D in reduced_partition[j]:
                    return False, [], set()

    all_derived = set().union(*derived_per_block)

    return (
    True,
    reduced_partition,
    all_derived
    )


# =========================================================
# GLOBAL SAP OPTIMIZATION
# =========================================================

def globally_optimal_partition(Gamma_input):

    Gamma_0 = set(Gamma_input)

    best_partition = None

    best_derived = set()

    best_cost = None

    checked = 0

    admissible_count = 0

    # enumerate ALL partitions

    for partition in all_partitions(list(Gamma_0)):

        checked += 1

        (
            admissible,
            reduced_partition,
            derived_sets
        ) = admissible_partition(
            partition,
            Gamma_0
        )

        if admissible:

            admissible_count += 1

            cost = EC_cost(
                reduced_partition
            )

            # global minimization

            if (
                best_cost is None
                or cost < best_cost
            ):

                best_cost = cost

                best_partition = reduced_partition

                best_derived = derived_sets

    return (
        best_partition,
        best_derived,
        best_cost,
        checked,
        admissible_count
    )


# =========================================================
# GENERATE ALL VALID Γ0
#
# Conditions:
# 1. no singleton sets
# 2. all 4 participants appear
# =========================================================

P = {1,2,3,4}

all_sets = []

for r in range(2,5):

    for c in combinations(sorted(P), r):

        all_sets.append(frozenset(c))


# =========================================================
# ANTICHAIN CONDITION
# =========================================================

def antichain(F):

    for i in range(len(F)):

        for j in range(i+1, len(F)):

            A = F[i]
            B = F[j]

            if A.issubset(B) or B.issubset(A):

                return False

    return True


# =========================================================
# GENERATE THE 87 STRUCTURES
# =========================================================

Gamma_family = []

for r in range(1, len(all_sets)+1):

    for comb in combinations(all_sets, r):

        comb = list(comb)

        if not antichain(comb):

            continue

        U = set()

        for A in comb:

            U |= set(A)

        # all 4 participants appear

        if U == P:

            Gamma_family.append(comb)


print("Total Γ0 =", len(Gamma_family))


# =========================================================
# RUN ALL 87 STRUCTURES
# =========================================================

rows = []

for idx, Gamma in enumerate(Gamma_family, start=1):

    print(f"Running {idx}/{len(Gamma_family)}")

    (
        result,
        derived,
        cost,
        checked,
        admissible_count
    ) = globally_optimal_partition(Gamma)

    # -----------------------------------------------------
    # stringify Γ0
    # -----------------------------------------------------

    gamma_str = []

    for A in sorted(Gamma, key=lambda x: sorted(list(x))):

        gamma_str.append(
            "{" + ",".join(map(str, sorted(A))) + "}"
        )

    gamma_repr = (
        "{"
        + ", ".join(gamma_str)
        + "}"
    )

    # -----------------------------------------------------
    # stringify optimal partition
    # -----------------------------------------------------

    partition_repr = []

    for block in result:

        block_str = []

        for A in sorted(block, key=lambda x: sorted(list(x))):

            block_str.append(
                "{" + ",".join(map(str, sorted(A))) + "}"
            )

        partition_repr.append(block_str)

    # -----------------------------------------------------
    # derived sets
    # -----------------------------------------------------

    derived_repr = []

    for D in sorted(derived, key=lambda x: sorted(list(x))):

        derived_repr.append(
            "{" + ",".join(map(str, sorted(D))) + "}"
        )

    rows.append({

        "Gamma_0": gamma_repr,

        "Optimal Partition": str(partition_repr),

        "Derived Sets": str(derived_repr),

        "Partitions Checked": checked,

        "SAP-Admissible Partitions": admissible_count,

        "Optimized Encoding Complexity": cost

    })


# =========================================================
# CREATE DATAFRAME
# =========================================================

df = pd.DataFrame(rows)


# =========================================================
# SAVE CSV
# =========================================================

df.to_csv(
    "SAP_Verification_87.csv",
    index=False
)

print("\nDONE\n")

print(df.head())


# =========================================================
# DOWNLOAD CSV (COLAB)
# =========================================================

from google.colab import files

files.download("SAP_Verification_87.csv")

Total Γ0 = 87
Running 1/87
Running 2/87
Running 3/87
Running 4/87
Running 5/87
Running 6/87
Running 7/87
Running 8/87
Running 9/87
Running 10/87
Running 11/87
Running 12/87
Running 13/87
Running 14/87
Running 15/87
Running 16/87
Running 17/87
Running 18/87
Running 19/87
Running 20/87
Running 21/87
Running 22/87
Running 23/87
Running 24/87
Running 25/87
Running 26/87
Running 27/87
Running 28/87
Running 29/87
Running 30/87
Running 31/87
Running 32/87
Running 33/87
Running 34/87
Running 35/87
Running 36/87
Running 37/87
Running 38/87
Running 39/87
Running 40/87
Running 41/87
Running 42/87
Running 43/87
Running 44/87
Running 45/87
Running 46/87
Running 47/87
Running 48/87
Running 49/87
Running 50/87
Running 51/87
Running 52/87
Running 53/87
Running 54/87
Running 55/87
Running 56/87
Running 57/87
Running 58/87
Running 59/87
Running 60/87
Running 61/87
Running 62/87
Running 63/87
Running 64/87
Running 65/87
Running 66/87
Running 67/87
Running 68/87
Running 69/87
Running 70/87
Running 71/87
R

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Partitiong verification on all 166 Access Stucture for statistical testing

In [5]:
from itertools import combinations
import pandas as pd


# =========================================================
# Local intersection
# =========================================================

def local_intersection(subfamily):

    return set.intersection(*map(set, subfamily))


# =========================================================
# Quotient residuals
# =========================================================

def quotient_residuals(subfamily, S):

    return [set(A) - S for A in subfamily]


# =========================================================
# k-SC condition
# =========================================================

def satisfies_kSC(residuals):

    k = len(residuals)

    for i in range(k):

        ok = True

        for j in range(k):

            if i == j:

                continue

            if residuals[i].intersection(residuals[j]):

                ok = False

                break

        if ok:

            return True

    return False


# =========================================================
# XOR-induced support over F2
# =========================================================

def xor_support(subfamily):

    count = {}

    for A in subfamily:

        for x in A:

            count[x] = count.get(x, 0) + 1

    B = set()

    for x in count:

        if count[x] % 2 == 1:

            B.add(x)

    return frozenset(B)




# =========================================================
# PARITY CLOSURE OF A BLOCK
# =========================================================

def closure_sets(block):

    derived = set()

    n = len(block)

    for k in range(3, n + 1, 2):

        for sub in combinations(block, k):

            X = xor_support(sub)

            if X not in block:
                derived.add(X)

    return derived


# =========================================================
# MONOTONE QUALIFICATION CHECK
# =========================================================

def monotone_qualified(B, Gamma_0):

    for A in Gamma_0:

        if set(A).issubset(set(B)):

            return True

    return False


# =========================================================
# PROPERTY A CHECK
# =========================================================

def admissible_family(family, Gamma_0):

    n = len(family)

    for k in range(3, n + 1, 2):

        for sub in combinations(family, k):

            S = local_intersection(sub)

            residuals = quotient_residuals(sub, S)

            # -------------------------------------------------
            # Property A failure
            # -------------------------------------------------

            if not satisfies_kSC(residuals):

                B = xor_support(sub)

                # -------------------------------------------------
                # Property B admissibility
                # -------------------------------------------------

                if not monotone_qualified(B, Gamma_0):

                    return False

    return True


# =========================================================
# CHECK WHETHER B IS GENERATED
# =========================================================

def generated_by_family(B, family):

    n = len(family)

    for k in range(3, n + 1, 2):

        for sub in combinations(family, k):

            X = xor_support(sub)

            if X == B:

                return True

    return False


# =========================================================
# MINIMAL AFFINE GENERATING BLOCK
# =========================================================

def reduce_block(block):

    generators = list(block)

    derived = set()

    changed = True

    while changed:

        changed = False

        for B in generators.copy():

            others = [
                A for A in generators
                if A != B
            ]

            if generated_by_family(B, others):

                generators.remove(B)

                derived.add(B)

                changed = True

                break

    return generators, derived


# =========================================================
# SUPPORT OF BLOCK
# =========================================================

def block_support(block):

    S = set()

    for A in block:

        S |= set(A)

    return S


# =========================================================
# Encoding COST
# =========================================================

def EC_cost(partition):

    total = 0

    for block in partition:

        total += len(block_support(block))

    return total


# =========================================================
# FORMAT BLOCK
# =========================================================

def format_block(block):

    formatted = []

    for A in sorted(block, key=lambda x: sorted(list(x))):

        formatted.append(
            "{" + ",".join(map(str, sorted(A))) + "}"
        )

    return formatted


# =========================================================
# GENERATE ALL SET PARTITIONS
# =========================================================

def all_partitions(collection):

    if len(collection) == 1:

        yield [collection]

        return

    first = collection[0]

    for smaller in all_partitions(collection[1:]):

        # insert into existing block

        for n, subset in enumerate(smaller):

            yield (
                smaller[:n]
                + [[first] + subset]
                + smaller[n + 1:]
            )

        # create singleton block

        yield [[first]] + smaller


# =========================================================
# CHECK GLOBAL SAP PARTITION
# =========================================================

def admissible_partition(partition, Gamma_0):

    reduced_partition = []
    derived_per_block = []

    for block in partition:

        if not admissible_family(block, Gamma_0):
            return False, [], set()


        reduced, _ = reduce_block(block)

        derived = closure_sets(reduced)

        reduced_partition.append(reduced)
        derived_per_block.append(derived)

# --------------------------------------------
# NEW CHECK
# Derived sets cannot appear in another block
# --------------------------------------------

    for i in range(len(reduced_partition)):

        for j in range(len(reduced_partition)):

            if i == j:
                continue

            for D in derived_per_block[i]:

                if D in reduced_partition[j]:
                    return False, [], set()

    all_derived = set().union(*derived_per_block)

    return (
    True,
    reduced_partition,
    all_derived
    )


# =========================================================
# GLOBAL SAP OPTIMIZATION
# =========================================================

def globally_optimal_partition(Gamma_input):

    Gamma_0 = set(Gamma_input)

    best_partition = None

    best_derived = set()

    best_cost = None

    checked = 0

    admissible_count = 0

    # enumerate ALL partitions

    for partition in all_partitions(list(Gamma_0)):

        checked += 1

        (
            admissible,
            reduced_partition,
            derived_sets
        ) = admissible_partition(
            partition,
            Gamma_0
        )

        if admissible:

            admissible_count += 1

            cost = EC_cost(
                reduced_partition
            )

            # global minimization

            if (
                best_cost is None
                or cost < best_cost
            ):

                best_cost = cost

                best_partition = reduced_partition

                best_derived = derived_sets

    return (
        best_partition,
        best_derived,
        best_cost,
        checked,
        admissible_count
    )


# =========================================================
# GENERATE ALL NONTRIVIAL Γ0
#
# Conditions:
# 1. singleton sets allowed
# 2. NO requirement that all 4 participants appear
# =========================================================

P = {1,2,3,4}

all_sets = []

# ---------------------------------------------------------
# all nonempty subsets
# ---------------------------------------------------------

for r in range(1,5):

    for c in combinations(sorted(P), r):

        all_sets.append(frozenset(c))


# =========================================================
# ANTICHAIN CONDITION
# =========================================================

def antichain(F):

    for i in range(len(F)):

        for j in range(i+1, len(F)):

            A = F[i]
            B = F[j]

            if A.issubset(B) or B.issubset(A):

                return False

    return True


# =========================================================
# GENERATE ALL 166 STRUCTURES
# =========================================================

Gamma_family = []

for r in range(1, len(all_sets)+1):

    for comb in combinations(all_sets, r):

        comb = list(comb)

        if not antichain(comb):

            continue

        # -------------------------------------------------
        # EXCLUDE EMPTY FAMILY ONLY
        # -------------------------------------------------

        Gamma_family.append(comb)


print("Total Γ0 =", len(Gamma_family))


# =========================================================
# RUN ALL 166 STRUCTURES
# =========================================================

rows = []

for idx, Gamma in enumerate(Gamma_family, start=1):

    print(f"Running {idx}/{len(Gamma_family)}")

    (
        result,
        derived,
        cost,
        checked,
        admissible_count
    ) = globally_optimal_partition(Gamma)

    # -----------------------------------------------------
    # stringify Γ0
    # -----------------------------------------------------

    gamma_str = []

    for A in sorted(Gamma, key=lambda x: sorted(list(x))):

        gamma_str.append(
            "{" + ",".join(map(str, sorted(A))) + "}"
        )

    gamma_repr = (
        "{"
        + ", ".join(gamma_str)
        + "}"
    )

    # -----------------------------------------------------
    # stringify optimal partition
    # -----------------------------------------------------

    partition_repr = []

    for block in result:

        block_str = []

        for A in sorted(block, key=lambda x: sorted(list(x))):

            block_str.append(
                "{" + ",".join(map(str, sorted(A))) + "}"
            )

        partition_repr.append(block_str)

    # -----------------------------------------------------
    # derived sets
    # -----------------------------------------------------

    derived_repr = []

    for D in sorted(derived, key=lambda x: sorted(list(x))):

        derived_repr.append(
            "{" + ",".join(map(str, sorted(D))) + "}"
        )

    rows.append({

        "Gamma_0": gamma_repr,

        "Optimal Partition": str(partition_repr),

        "Derived Sets": str(derived_repr),

        "Partitions Checked": checked,

        "SAP-Admissible Partitions": admissible_count,

        "Optimized Encoding Complexity": cost

    })


# =========================================================
# CREATE DATAFRAME
# =========================================================

df = pd.DataFrame(rows)


# =========================================================
# SAVE CSV
# =========================================================

df.to_csv(
    "SAP_Verification_all.csv",
    index=False
)

print("\nDONE\n")

print(df.head())


# =========================================================
# DOWNLOAD CSV (COLAB)
# =========================================================

from google.colab import files

files.download("SAP_Verification_all.csv")

Total Γ0 = 166
Running 1/166
Running 2/166
Running 3/166
Running 4/166
Running 5/166
Running 6/166
Running 7/166
Running 8/166
Running 9/166
Running 10/166
Running 11/166
Running 12/166
Running 13/166
Running 14/166
Running 15/166
Running 16/166
Running 17/166
Running 18/166
Running 19/166
Running 20/166
Running 21/166
Running 22/166
Running 23/166
Running 24/166
Running 25/166
Running 26/166
Running 27/166
Running 28/166
Running 29/166
Running 30/166
Running 31/166
Running 32/166
Running 33/166
Running 34/166
Running 35/166
Running 36/166
Running 37/166
Running 38/166
Running 39/166
Running 40/166
Running 41/166
Running 42/166
Running 43/166
Running 44/166
Running 45/166
Running 46/166
Running 47/166
Running 48/166
Running 49/166
Running 50/166
Running 51/166
Running 52/166
Running 53/166
Running 54/166
Running 55/166
Running 56/166
Running 57/166
Running 58/166
Running 59/166
Running 60/166
Running 61/166
Running 62/166
Running 63/166
Running 64/166
Running 65/166
Running 66/166
Runn

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>